# Internship Project: Unemployment Analysis in India

## Introduction

This notebook presents an analysis of unemployment data in India, a project undertaken as part of my internship. The goal of this project is to explore trends in unemployment rates, identify key regions affected, and analyze the impact of significant events, such as the COVID-19 pandemic, on employment statistics.

### Project Objectives:
- Load and preprocess unemployment datasets.
- Provide an overview and summary statistics of the data.
- Analyze regional unemployment trends.
- Investigate the impact of the COVID-19 era on employment.

Developed by: [Your Name/Intern ID]
Date: [Current Date]

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Optional

import pandas as pd


DATA_FOLDER = Path('/content/') # Updated to the correct path where files are located
FIRST_FILE = DATA_FOLDER / 'Unemployment in India.csv'
SECOND_FILE = DATA_FOLDER / 'Unemployment_Rate_upto_11_2020.csv'


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.strip() for col in df.columns]
    return df


def load_unemployment_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = clean_columns(df)
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    return df


def print_dataset_overview(df: pd.DataFrame, title: str) -> None:
    print('=' * 80)
    print(title)
    print('=' * 80)
    print(f'Records: {len(df):,}')
    print(f'Columns: {len(df.columns):,}')
    print(df.columns.tolist())
    print('\nFirst 5 rows:')
    print(df.head(5).to_string(index=False))
    print('\n')


def unemployment_summary(df: pd.DataFrame, value_column: str = 'Estimated Unemployment Rate (%)') -> None:
    if value_column not in df.columns:
        print(f'Column not found: {value_column}')
        return

    valid = df[[value_column, 'Region']].dropna()
    print('Global summary statistics for unemployment rate:')
    print(valid[value_column].describe().to_string())
    print('\nTop 10 regions by mean unemployment rate:')
    print(
        valid.groupby('Region')[value_column]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .round(2)
        .to_string()
    )
    print('\nTop 10 regions by minimum unemployment rate:')
    print(
        valid.groupby('Region')[value_column]
        .min()
        .sort_values(ascending=True)
        .head(10)
        .round(2)
        .to_string()
    )
    print('\nTop 10 regions by maximum unemployment rate:')
    print(
        valid.groupby('Region')[value_column]
        .max()
        .sort_values(ascending=False)
        .head(10)
        .round(2)
        .to_string()
    )
    print('\n')


def covid_era_analysis(df: pd.DataFrame) -> None:
    if 'Date' not in df.columns or 'Region' not in df.columns:
        print('Dataset is missing Date or Region column. Skipping COVID-era analysis.')
        return

    if df['Date'].isna().all():
        print('No parseable dates found; COVID-era analysis cannot continue.')
        return

    covid_start = pd.Timestamp('2020-03-01')
    covid_df = df[df['Date'] >= covid_start].copy()
    if covid_df.empty:
        print('No records found in the COVID period (2020-03 onward).')
        return

    print('COVID-era unemployment analysis (from March 2020 onward):')
    print(f'Total COVID-period records: {len(covid_df):,}')
    print('Monthly unemployment rate changes by region:')
    aggregated = (
        covid_df.groupby('Region')['Estimated Unemployment Rate (%)']
        .agg(['mean', 'min', 'max', 'std'])
        .sort_values(by='mean', ascending=False)
        .round(2)
    )
    print(aggregated.head(10).to_string())
    print('\n')


def top_regions_in_period(df: pd.DataFrame, start_date: Optional[pd.Timestamp], end_date: Optional[pd.Timestamp]) -> None:
    if 'Date' not in df.columns:
        return

    period_df = df
    if start_date is not None:
        period_df = period_df[period_df['Date'] >= start_date]
    if end_date is not None:
        period_df = period_df[period_df['Date'] <= end_date]

    if period_df.empty:
        print('No records matched the selected period.')
        return

    print(f'Top 10 regions by average unemployment rate between {start_date.date() if start_date else "start"} and {end_date.date() if end_date else "end"}:')
    top10 = (
        period_df.groupby('Region')['Estimated Unemployment Rate (%)']
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .round(2)
    )
    print(top10.to_string())
    print('\n')


def main() -> None:
    print('Loading datasets...')
    first_df = load_unemployment_data(FIRST_FILE)
    second_df = load_unemployment_data(SECOND_FILE)

    print_dataset_overview(first_df, 'Dataset 1: Unemployment in India')
    print_dataset_overview(second_df, 'Dataset 2: Unemployment Rate up to 11/2020')

    print('Summary for Dataset 1')
    unemployment_summary(first_df)
    print('Summary for Dataset 2')
    unemployment_summary(second_df)

    covid_era_analysis(second_df)
    top_regions_in_period(second_df, pd.Timestamp('2020-03-01'), pd.Timestamp('2020-11-30'))

    print('Analysis complete.')


if __name__ == '__main__':
    main()

Loading datasets...
Dataset 1: Unemployment in India
Records: 768
Columns: 7
['Region', 'Date', 'Frequency', 'Estimated Unemployment Rate (%)', 'Estimated Employed', 'Estimated Labour Participation Rate (%)', 'Area']

First 5 rows:
        Region       Date Frequency  Estimated Unemployment Rate (%)  Estimated Employed  Estimated Labour Participation Rate (%)  Area
Andhra Pradesh 2019-05-31   Monthly                             3.65          11999139.0                                    43.24 Rural
Andhra Pradesh 2019-06-30   Monthly                             3.05          11755881.0                                    42.05 Rural
Andhra Pradesh 2019-07-31   Monthly                             3.75          12086707.0                                    43.50 Rural
Andhra Pradesh 2019-08-31   Monthly                             3.32          12285693.0                                    43.97 Rural
Andhra Pradesh 2019-09-30   Monthly                             5.17          12256762.0